## Reading final_data

In [1]:
final_data = spark.read.csv(
    "hdfs://ubuntu-master:9000/user/ubuntu/Downloads/final_data.csv",
    header=True,
    inferSchema=False,
    encoding="UTF-8"
)
final_data.show(5)

+--------+--------------------+-----------------+-----------+-------------+-------+--------+------+----------+-----------+----------------+----------+-------+--------+
| GRID_CD|            geometry|senior_population|new_village|senior_center|medical|pharmacy|market|restaurant|bus_station|service_facility|enterprise|factory|운행대수|
+--------+--------------------+-----------------+-----------+-------------+-------+--------+------+----------+-----------+----------------+----------+-------+--------+
|다아3700|POLYGON ((126.782...|              0.0|          0|            0|      0|       0|     0|         0|          0|               0|         0|      0|       0|
|다아3800|POLYGON ((126.793...|              0.0|          0|            0|      0|       0|     0|         0|          0|               0|         0|      0|       0|
|다아3900|POLYGON ((126.805...|              0.0|          0|            0|      0|       0|     0|         0|          0|               0|         0|      0|       0|
|다

## Installing required packages

In [2]:
#pip install xgboost --break-system-packages

In [3]:
#pip install scikit-learn --break-system-packages

## XGBoost

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from xgboost.spark import SparkXGBClassifier

In [ ]:
# Column Type Conversion (Manual casting since inferSchema=False was used)
exclude_cols = ["GRID_CD", "geometry", "bus", "운행대수", "predict_proba", "proba_interval"]

# Create a binary variable 'bus' from '운행대수'
final_data = final_data.withColumn(
    "bus",
    F.when(F.col("운행대수").cast(DoubleType()) > 0, 1).otherwise(0).cast(IntegerType())
)

# Extract numeric columns (excluding specified columns)
all_cols = final_data.columns
feature_cols = [c for c in all_cols if c not in exclude_cols]

In [ ]:
# Cast from String to Double
for c in feature_cols:
    final_data = final_data.withColumn(c, F.col(c).cast(DoubleType()))

# Missing Value Imputation (Replaced with the median value)
from pyspark.ml.feature import Imputer

imputer = Imputer(
    inputCols=feature_cols,
    outputCols=feature_cols,
    strategy="median"
)
final_data = imputer.fit(final_data).transform(final_data)

# Feature Vector Generation
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)
final_data = assembler.transform(final_data)

In [ ]:
# Train/Test Split (8:2 ratio, stratification is handled manually in PySpark)
bus1 = final_data.filter(F.col("bus") == 1)
bus0 = final_data.filter(F.col("bus") == 0)

train1, test1 = bus1.randomSplit([0.8, 0.2], seed=42)
train0, test0 = bus0.randomSplit([0.8, 0.2], seed=42)

train_df = train1.union(train0)
test_df  = test1.union(test0)

# SparkXGBClassifier Training
xgb_model = SparkXGBClassifier(
    features_col="features",
    label_col="bus",
    pred_contrib_col="predict_proba",
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss"
)
xgb_fitted = xgb_model.fit(train_df)

2026-06-08 22:41:21,076 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'colsample_bytree': 0.8, 'device': 'cpu', 'eval_metric': 'logloss', 'learning_rate': 0.05, 'max_depth': 4, 'subsample': 0.8, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 300}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-06-08 22:41:30,717 INFO XGBoost-PySpark: _train_booster Training on CPUs 1]
[22:41:31] Task 0 got rank 0
[22:41:31] [0]	training-logloss:0.55627
[22:41:31] [1]	training-logloss:0.55410
[22:41:31] [2]	training-logloss:0.54843
[22:41:31] [3]	training-logloss:0.54378
[22:41:32] [4]	training-logloss:0.53972
[22:41:32] [5]	training-logloss:0.53624
[22:41:32] [6]	training-logloss:0.53327
[22:41:32] [7]	training-logloss:0.53146
[22:41:32] [8]	training-logloss:0.52861
[22:41:32] [9]	training-logloss:0.52777
[22:41:32] [10]	training-logloss:0.52633
[22:41:32] [11]	training-logloss:0.52417
[22:41

In [ ]:
# Model Performance Evaluation
predictions = xgb_fitted.transform(test_df)

binary_eval = BinaryClassificationEvaluator(
    labelCol="bus",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
multi_eval = MulticlassClassificationEvaluator(
    labelCol="bus",
    predictionCol="prediction"
)

print("XGBoost Model Performance")
print(f"ROC-AUC   : {binary_eval.evaluate(predictions):.4f}")
print(f"Accuracy  : {multi_eval.setMetricName('accuracy').evaluate(predictions):.4f}")
print(f"Precision : {multi_eval.setMetricName('weightedPrecision').evaluate(predictions):.4f}")
print(f"Recall    : {multi_eval.setMetricName('weightedRecall').evaluate(predictions):.4f}")
print(f"F1-score  : {multi_eval.setMetricName('f1').evaluate(predictions):.4f}")

XGBoost Model Performance


2026-06-08 22:42:39,824 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
                                                                                

ROC-AUC   : 0.6614


2026-06-08 22:42:47,501 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
                                                                                

Accuracy  : 0.7865


...

Precision : 0.7756


2026-06-08 22:42:51,571 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
                                                                                

Recall    : 0.7865


2026-06-08 22:42:54,577 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
[Stage 28:=============================>                            (1 + 1) / 2]

F1-score  : 0.7291


In [ ]:
# Prediction Probability Calculation for the Entire Grid
all_predictions = xgb_fitted.transform(final_data)

# Extract positive class (bus=1) probability from the probability column
extract_proba = F.udf(lambda v: float(v[1]), DoubleType())
all_predictions = all_predictions.withColumn(
    "predict_proba",
    extract_proba(F.col("probability"))
)

print("\n사용 Feature")
print(feature_cols)

all_predictions.select("GRID_CD", "bus", "predict_proba").show(5)


사용 Feature
['senior_population', 'new_village', 'senior_center', 'medical', 'pharmacy', 'market', 'restaurant', 'bus_station', 'service_facility', 'enterprise', 'factory']
+--------+---+-------------------+
| GRID_CD|bus|      predict_proba|
+--------+---+-------------------+
|다아3700|  0|0.16862502694129944|
|다아3800|  0|0.16862502694129944|
|다아3900|  0|0.16862502694129944|
|다아4000|  0|0.16862502694129944|
|다아4001|  0|0.16862502694129944|
+--------+---+-------------------+
only showing top 5 rows


2026-06-08 22:43:23,225 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs


## Ranking of New Candidate Locations

In [ ]:
from pyspark.sql import functions as F

# Filter only new candidate locations where bus = 0
candidate_df = all_predictions.filter(F.col("bus") == 0)

In [ ]:
# Sort in descending order by prediction probability + Add rank column
candidate_rank = candidate_df.orderBy(F.col("predict_proba").desc())

candidate_rank = candidate_rank.withColumn(
    "rank",
    F.row_number().over(
        __import__("pyspark.sql.window", fromlist=["Window"]).Window.orderBy(F.col("predict_proba").desc())
    )
)

# Configure readable column ordering
result_cols = ["rank", "GRID_CD", "predict_proba"] + feature_cols
candidate_rank_result = candidate_rank.select(result_cols)

# Display the top 30 rows
TOP_N = 30
candidate_rank_result.show(TOP_N, truncate=False)

26/06/08 22:46:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/08 22:46:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/08 22:46:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
2026-06-08 22:46:53,058 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs


+----+--------+------------------+-----------------+-----------+-------------+-------+--------+------+----------+-----------+----------------+----------+-------+
|rank|GRID_CD |predict_proba     |senior_population|new_village|senior_center|medical|pharmacy|market|restaurant|bus_station|service_facility|enterprise|factory|
+----+--------+------------------+-----------------+-----------+-------------+-------+--------+------+----------+-----------+----------------+----------+-------+
|1   |다바7987|0.955355167388916 |0.0              |0.0        |3.0          |0.0    |0.0     |1.0   |33.0      |4.0        |0.0             |205.0     |3.0    |
|2   |다사6945|0.954929530620575 |7299.0           |1.0        |9.0          |1.0    |8.0     |0.0   |100.0     |2.0        |0.0             |0.0       |0.0    |
|3   |다사7351|0.9149342775344849|9955.0           |0.0        |4.0          |2.0    |4.0     |0.0   |219.0     |4.0        |0.0             |0.0       |0.0    |
|4   |다사7563|0.8769282698631287|0.

In [ ]:
# Save to HDFS
HDFS_SAVE_PATH = "hdfs://ubuntu-master:9000/user/ubuntu/Downloads/xgboost_optimal_location_ranking1.csv"
candidate_rank_result.coalesce(1).write.csv(
    HDFS_SAVE_PATH,
    header=True,
    mode="overwrite",
    encoding="UTF-8"
)
print(f"HDFS 저장 완료: {HDFS_SAVE_PATH}")

26/06/08 22:51:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/08 22:51:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/08 22:51:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
2026-06-08 22:51:21,151 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
2026-06-08 22:51:26,926 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
26/06/08 22:51:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/08 22:51:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
2

HDFS 저장 완료: hdfs://ubuntu-master:9000/user/ubuntu/Downloads/xgboost_optimal_location_ranking1.csv


## Visualization

In [ ]:
import folium
import geopandas as gpd
import pandas as pd
from shapely import wkt

# Spark DataFrame → Pandas Conversion
map_df = all_predictions.select(
    "GRID_CD", "bus", "predict_proba", "geometry"
).toPandas()

map_df["geometry"] = map_df["geometry"].apply(wkt.loads)

gdf = gpd.GeoDataFrame(map_df, geometry="geometry", crs="EPSG:4326")

# Binning predict_proba
bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
labels = [
    "0.0-0.2",
    "0.2-0.4",
    "0.4-0.6",
    "0.6-0.8",
    "0.8-1.0"
]

gdf["proba_interval"] = pd.cut(
    gdf["predict_proba"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# Red Color Density Palette
red_color_dict = {
    "0.0-0.2": "#fee5d9",
    "0.2-0.4": "#fcae91",
    "0.4-0.6": "#fb6a4a",
    "0.6-0.8": "#de2d26",
    "0.8-1.0": "#a50f15"
}

existing_color = "#87CEEB"   # Existing installation area: Sky blue

# Map Center Configuration
center_lat = gdf.geometry.centroid.y.mean()
center_lon = gdf.geometry.centroid.x.mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles="cartodbpositron"
)

# Style Function
def style_function(feature):
    props = feature["properties"]

    # Sky blue for existing installation areas
    if props["bus"] == 1:
        return {
            "fillColor": existing_color,
            "color": "gray",
            "weight": 0.4,
            "fillOpacity": 0.75
        }
    # Apply red color density to new candidate areas based on predict_proba intervals
    interval = props.get("proba_interval")
    color = red_color_dict.get(interval, "#ffffff")

    return {
        "fillColor": color,
        "color": "gray",
        "weight": 0.3,
        "fillOpacity": 0.70
    }

# Add Grid to Map
folium.GeoJson(
    gdf,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=["GRID_CD", "bus", "predict_proba", "proba_interval"],
        aliases=["GRID_CD", "기존 설치 여부", "예측 확률", "확률 구간"],
        localize=True
    )
).add_to(m)

# Add Legend
legend_html = """
<div style="
position: fixed;
bottom: 50px; left: 50px; width: 280px;
background-color: white;
border:2px solid grey;
z-index:9999;
font-size:14px;
padding: 12px;
line-height: 1.6;
box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
">
<b>Legend</b><br>

<span style="background-color:#87CEEB; display:inline-block; width:18px; height:12px; border:1px solid #999;"></span>
기존 설치 지역<br><br>

<b>신규 후보 지역 예측 확률</b><br>

<span style="background-color:#fee5d9; display:inline-block; width:18px; height:12px; border:1px solid #999;"></span>
0.0 ~ 0.2 : 매우 낮음<br>

<span style="background-color:#fcae91; display:inline-block; width:18px; height:12px; border:1px solid #999;"></span>
0.2 ~ 0.4 : 낮음<br>

<span style="background-color:#fb6a4a; display:inline-block; width:18px; height:12px; border:1px solid #999;"></span>
0.4 ~ 0.6 : 보통<br>

<span style="background-color:#de2d26; display:inline-block; width:18px; height:12px; border:1px solid #999;"></span>
0.6 ~ 0.8 : 높음<br>

<span style="background-color:#a50f15; display:inline-block; width:18px; height:12px; border:1px solid #999;"></span>
0.8 ~ 1.0 : 매우 높음<br>

</div>
"""

m

2026-06-08 22:54:29,454 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
/tmp/ipykernel_6879/2004029093.py:44: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lat = gdf.geometry.centroid.y.mean()
/tmp/ipykernel_6879/2004029093.py:45: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lon = gdf.geometry.centroid.x.mean()


XGBoost 최적 입지 지도 저장 완료: /content/drive/MyDrive/xgboost_optimal_location_map.html
